# Web3 Python 100本ノック：第1章
## §1-10 (Aコース)：ウォレット資産を自動集計し、PDFレポートを生成せよ

金融や投資、資産管理ツールに興味がある方向けの最終課題です。

これまでに学んだ「オンチェーンデータの取得（ETH・USDT）」「単位変換」「外部API（CoinGecko）での法定通貨レート取得」のすべてを組み合わせます。
指定したアドレスの総資産をリアルタイムに計算し、さらにPythonのライブラリを使って **「1枚のPDFレポート」として自動出力する** スクリプトを完成させましょう。

> **注意**: 各セルを順番に実行してください。

### 準備

In [8]:
# web3 に加えて、PDF生成用の fpdf ライブラリをインストールします
!pip install web3==7.16.0 requests fpdf matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached numpy-2.5.1-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 9.5/9.5 MB 51.5 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 58.9 MB/s  0:00:00
Using cached numpy-2.5.1-cp314-cp314-win_amd64.whl (12.6 MB)
Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl (7.2 MB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)

   ---------------------------------------- 0/8 [pyparsing]
   ----- ---------------------------------- 1/8 [pillow]
   ----- ---------------------------------- 1/8 [pillow]
   ----- -------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## ブロックチェーンとドキュメント生成

取得したデータをただ画面に出すのではなく、`fpdf` を使ってファイルに書き出します。
このスクリプトを応用すれば、「毎朝8時に自社の保有資産を計算し、PDFにしてチームにメール送信する」といった実用的な経理・投資管理Botを作ることができます。

In [11]:
from web3 import Web3
import requests
from fpdf import FPDF
import matplotlib.pyplot as plt
from datetime import datetime

# --- 1. Web3 接続設定 ---
RPC_URL = "https://eth.drpc.org"
headers = {'User-Agent': 'Mozilla/5.0'}
w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={'headers': headers, 'timeout': 5}))


### データ取得
ターゲットアドレスを変えることで自身のwalletなども対応できます。

In [9]:
print("資産データの集計を開始")

# ターゲットアドレス（例としてVitalik氏のアドレス）
target_address = w3.to_checksum_address("0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045")

資産データの集計を開始


In [ ]:
try:
    # --- 2. オンチェーンデータの取得 ---
    eth_bal_wei = w3.eth.get_balance(target_address)
    eth_bal = float(w3.from_wei(eth_bal_wei, "ether"))
    
    USDT_ADDRESS = w3.to_checksum_address("0xdAC17F958D2ee523a2206206994597C13D831ec7")
    erc20_abi = [{"constant": True, "inputs": [{"name": "_owner", "type": "address"}], "name": "balanceOf", "outputs": [{"name": "balance", "type": "uint256"}], "type": "function"}]
    usdt_contract = w3.eth.contract(address=USDT_ADDRESS, abi=erc20_abi)
    usdt_bal_raw = usdt_contract.functions.balanceOf(target_address).call()
    usdt_bal = usdt_bal_raw / (10 ** 6)
    
    # --- 3. オフチェーン（為替レートと時系列）データの取得 ---
    print("現在のレートと7日間の時系列データを取得中...")
    
    # 現在の価格
    rates_url = "https://api.coingecko.com/api/v3/simple/price?ids=ethereum,tether&vs_currencies=usd,jpy"
    rates = requests.get(rates_url, timeout=10).json()
    eth_usd = rates["ethereum"]["usd"]
    usdt_usd = rates["tether"]["usd"]
    
    # 資産ごとのUSD価値
    eth_usd_value = eth_bal * eth_usd
    usdt_usd_value = usdt_bal * usdt_usd
    total_usd = eth_usd_value + usdt_usd_value

    # ETHの過去7日間の価格トレンド（CoinGecko market_chart API）
    trend_url = "https://api.coingecko.com/api/v3/coins/ethereum/market_chart?vs_currency=usd&days=7"
    trend_data = requests.get(trend_url, timeout=10).json()
    
    # タイムスタンプと価格をリストに抽出
    timestamps = [datetime.fromtimestamp(p[0] / 1000) for p in trend_data['prices']]
    prices = [p[1] for p in trend_data['prices']]

    # --- 4. チャート（画像）の生成と保存 ---
    print("チャートを描画中...")
    
    # [チャートA] 保有割合の円グラフ
    plt.figure(figsize=(4, 4))
    labels = ['Ethereum (ETH)', 'Tether (USDT)']
    sizes = [eth_usd_value, usdt_usd_value]
    colors = ['#627EEA', '#26A17B'] # ETHとUSDTのブランドカラー
    
    # 資産が0の場合はエラーを回避
    if sum(sizes) > 0:
        plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors)
    else:
        plt.text(0.5, 0.5, 'No Assets', horizontalalignment='center', verticalalignment='center')
        
    plt.title('Asset Allocation (USD)')
    plt.tight_layout()
    plt.savefig('allocation_chart.png', dpi=150)
    plt.close()

    # [チャートB] ETH 7日間価格トレンドの折れ線グラフ
    plt.figure(figsize=(7, 3))
    plt.plot(timestamps, prices, color='#627EEA', linewidth=2)
    plt.title('ETH 7-Day Price Trend (USD)')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig('trend_chart.png', dpi=150)
    plt.close()

    # --- 5. PDFレポートの生成 ---
    print("PDFレポートを構築中...")
    pdf = FPDF()
    pdf.set_auto_page_break(auto=False)
    pdf.add_page()
    
    # ヘッダー
    pdf.set_font("Arial", "B", 22)
    pdf.cell(0, 12, "Web3 Portfolio Report", ln=True, align="C")
    
    pdf.set_font("Arial", "I", 10)
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    pdf.cell(0, 6, f"Generated on: {current_time}", ln=True, align="C")
    pdf.ln(5) # 行間を圧縮
    
    # アドレス情報
    pdf.set_font("Arial", "B", 12)
    pdf.cell(0, 8, "Target Wallet Address:", ln=True)
    pdf.set_font("Arial", "", 10)
    pdf.cell(0, 6, target_address, ln=True)
    pdf.ln(5)
    
    # 資産詳細（テキスト）
    pdf.set_font("Arial", "B", 12)
    pdf.cell(0, 8, "Asset Breakdown:", ln=True)
    pdf.set_font("Arial", "", 12)
    pdf.cell(0, 6, f"- Ethereum (ETH) : {eth_bal:,.2f} ETH  (Current Rate: ${eth_usd:,.2f})", ln=True)
    pdf.cell(0, 6, f"- Tether (USDT)  : {usdt_bal:,.2f} USDT  (Current Rate: ${usdt_usd:,.2f})", ln=True)
    pdf.ln(5)
    
    # 総資産額（ハイライト）
    pdf.set_font("Arial", "B", 14)
    pdf.cell(0, 8, "Total Estimated Value:", ln=True)
    pdf.set_font("Arial", "B", 24)
    pdf.set_text_color(32, 129, 226) # Blue
    pdf.cell(0, 12, f"$ {total_usd:,.2f} USD", ln=True)
    pdf.set_text_color(0, 0, 0) # Reset color
    
    # 画像の挿入位置をミリ単位で調整
    # テキストが上部100mm程度で終わるため、Y=105からグラフを配置します
    pdf.image('allocation_chart.png', x=65, y=105, w=80) # 円グラフを中央寄りに
    pdf.image('trend_chart.png', x=15, y=190, w=180)     # トレンドグラフを横幅いっぱいに
    
    # フッター (ページ下端から上へ20mmの位置に固定)
    pdf.set_y(-20)
    pdf.set_font("Arial", "I", 8)
    pdf.set_text_color(128, 128, 128)
    pdf.cell(0, 6, "Powered by Web3.py, matplotlib & CoinGecko API", ln=True, align="R")
    pdf.cell(0, 6, "Generated by Cearasu.com", ln=True, align="R")
    
    # PDFの保存
    pdf_filename = "portfolio_report_pro.pdf"
    pdf.output(pdf_filename)
    
    print("=" * 40)
    print(f"成功: '{pdf_filename}' が生成されました！")
    print("=" * 40)

except Exception as e:
    print(f"エラーが発生しました: {e}")

現在のレートと7日間の時系列データを取得中...
チャートを描画中...
PDFレポートを構築中...
成功: 'portfolio_report_pro.pdf' が生成されました！


### ダウンロード(colabの場合)

In [ ]:
## 生成したpdfのダウンロード
from google.colab import files

files.download(pdf_filename)